### RLC Results to LaTex Table

In [ ]:
import pandas as pd

# 1. Load RLC main and holes-specific results
df  = pd.read_pickle('RLC_Results.pkl')

# 4. Classify variants into four groups
def classify_variant(r):
    v, tm = r['variant'], r['test_mode']
    if v == 'all':
        return 'all'
    elif v == 'all_holes':
        return 'all_holes'
    elif v == tm:
        return 'mode'
    elif v.isdigit() and v != tm:
        return 'other'
    else:
        return None

df['variant_group'] = df.apply(classify_variant, axis=1)
df_v = df.dropna(subset=['variant_group'])

# 5. Best config per system × {all, all_holes, mode}
best = (
    df_v[df_v['variant_group'].isin(['all','all_holes','mode'])]
    .groupby(['test_system','variant_group','model','hidden_size'])['auc']
    .mean().reset_index()
    .sort_values(['test_system','variant_group','auc'], ascending=[True,True,False])
    .groupby(['test_system','variant_group']).first().reset_index()
)

# 6. Fault mapping and comparison name
comp_map = {'OK_vs_C':'CAP', 'OK_vs_I':'IND', 'OK_vs_R':'RES'}
cmp_list = ['OK_vs_C','OK_vs_I','OK_vs_R']

# 7. Build rows with deltas at positions 4 and 7
rows = []
for sys in sorted(df_v['test_system'].unique()):
    cfg_all   = best[(best['test_system']==sys)&(best['variant_group']=='all')].iloc[0]
    cfg_holes = best[(best['test_system']==sys)&(best['variant_group']=='all_holes')].iloc[0]
    cfg_mode  = best[(best['test_system']==sys)&(best['variant_group']=='mode')].iloc[0]
    for comp in cmp_list:
        label = comp_map[comp]
        # aggregate function
        def agg(cfg, group):
            sub = df_v[
                (df_v['test_system']==sys) &
                (df_v['model']==cfg['model']) &
                (df_v['hidden_size']==cfg['hidden_size']) &
                (df_v['variant_group']==group) &
                (df_v['comparison']==comp)
            ]
            return sub['auc'].mean(), sub['auc'].std()
        m_all,   s_all   = agg(cfg_all,   'all')
        m_holes, s_holes = agg(cfg_holes, 'all_holes')
        m_mode,  s_mode  = agg(cfg_mode,  'mode')
        m_oth,   s_oth   = agg(cfg_mode,  'other')
        # deltas
        delta_h   = m_holes - m_all
        delta_n   = m_oth   - m_mode
        # formatted strings
        txt_all   = f"{m_all:.3f} $\\pm$ {s_all:.3f}"
        txt_holes = f"{m_holes:.3f} $\\pm$ {s_holes:.3f}"
        txt_mode  = f"{m_mode:.3f} $\\pm$ {s_mode:.3f}"
        txt_oth   = f"{m_oth:.3f} $\\pm$ {s_oth:.3f}"
        txt_dh    = f"{delta_h:+.3f}"
        txt_dn    = f"{delta_n:+.3f}"
        # bold Type vs Instance
        if m_all >= m_mode:
            txt_all = f"\\textbf{{{txt_all}}}"
        else:
            txt_mode = f"\\textbf{{{txt_mode}}}"
        rows.append((sys, label, txt_all, txt_holes, txt_dh, txt_mode, txt_oth, txt_dn))

# 8. Generate LaTeX
lines = [
    r"\begin{table*}[ht]",
    r"\centering",
    r"\caption{Caption}",
    r"\label{tab:rlc_deltas}",
    r"\begin{tabular}{cccccccc}",
    r"\toprule",
    r"\multirow{2}{*}{\textbf{S}} & \multirow{2}{*}{\textbf{Fault}}"
      r" & \multicolumn{1}{c}{\textbf{Type}} & \multicolumn{1}{c}{\textbf{Holes}}"
      r" & \multicolumn{1}{c}{\textbf{HolesType}} & \multicolumn{1}{c}{\textbf{Instance}}"
      r" & \multicolumn{1}{c}{\textbf{Non-Instance}} & \multicolumn{1}{c}{\textbf{DeltaNonInst-Inst}} \\",
    r"\cmidrule(lr){3-3} \cmidrule(lr){4-4} \cmidrule(lr){5-5} "
      r"\cmidrule(lr){6-6} \cmidrule(lr){7-7} \cmidrule(lr){8-8}",
    r" &  & \textit{mean $\pm$ std} & \textit{mean $\pm$ std} & & "
      r"\textit{mean $\pm$ std} & \textit{mean $\pm$ std} & \\",
    r"\midrule"
]
current_s = None
for sys, fault, a, h, dh, inst, oth, dn in rows:
    if sys != current_s:
        if current_s is not None:
            lines.append(r"\midrule")
        current_s = sys
        prefix = rf"\multirow{{3}}{{*}}{{{sys}}}"
    else:
        prefix = " " * len(rf"\multirow{{3}}{{*}}{{{sys}}}")
    lines.append(f"{prefix} & {fault} & {a} & {h} & {dh} & {inst} & {oth} & {dn} \\\\")
lines += [
    r"\bottomrule",
    r"\end{tabular}",
    r"\end{table*}"
]
latex_code = "\n".join(lines)
print(latex_code)


### ODE Results to LaTex Table

In [ ]:
import pandas as pd

# 1. Load the files
df  = pd.read_pickle('ODE_Results.pkl')

# 4. Classify variant groups (compare numeric variants as strings)
def classify_variant(r):
    v, tm = r['variant'], str(r['test_mode'])
    if v == 'all':
        return 'all'
    elif v == 'all_holes':
        return 'all_holes'
    elif v == tm:
        return 'mode'
    elif v.isdigit() and v != tm:
        return 'other'
    else:
        return None

df['variant_group'] = df.apply(classify_variant, axis=1)
df_v = df.dropna(subset=['variant_group'])

# 5. Find best (model, hidden_size) per system × {all, all_holes, mode}
best = (
    df_v[df_v['variant_group'].isin(['all','all_holes','mode'])]
    .groupby(['test_system','variant_group','model','hidden_size'])['auc']
    .mean()
    .reset_index()
    .sort_values(['test_system','variant_group','auc'], ascending=[True,True,False])
    .groupby(['test_system','variant_group'])
    .first()
    .reset_index()
)

# 6. Name of the single comparison
cmp = "OK_vs_anom"

# 7. Aggregate and compute deltas, placing them as 4th and 7th columns
rows = []
for sys in sorted(df_v['test_system'].unique()):
    cfg_all   = best[(best['test_system']==sys) & (best['variant_group']=='all')].iloc[0]
    cfg_holes = best[(best['test_system']==sys) & (best['variant_group']=='all_holes')].iloc[0]
    cfg_mode  = best[(best['test_system']==sys) & (best['variant_group']=='mode')].iloc[0]

    def agg(cfg, group):
        sub = df_v[
            (df_v['test_system']==sys) &
            (df_v['model']==cfg['model']) &
            (df_v['hidden_size']==cfg['hidden_size']) &
            (df_v['variant_group']==group) &
            (df_v['comparison']==cmp)
        ]
        return sub['auc'].mean(), sub['auc'].std()

    m_all,   s_all   = agg(cfg_all,   'all')
    m_holes, s_holes = agg(cfg_holes, 'all_holes')
    m_mode,  s_mode  = agg(cfg_mode,  'mode')
    m_oth,   s_oth   = agg(cfg_mode,  'other')

    # compute deltas
    delta_holes = m_holes - m_all
    delta_noninst = m_oth - m_mode

    # format mean±std and deltas
    txt_all     = f"{m_all:.3f} $\\pm$ {s_all:.3f}"
    txt_holes   = f"{m_holes:.3f} $\\pm$ {s_holes:.3f}"
    txt_mode    = f"{m_mode:.3f} $\\pm$ {s_mode:.3f}"
    txt_other   = f"{m_oth:.3f} $\\pm$ {s_oth:.3f}"
    txt_dh      = f"{delta_holes:+.3f}"
    txt_do      = f"{delta_noninst:+.3f}"

    # bold higher of Type vs Instance
    if m_all >= m_mode:
        txt_all = f"\\textbf{{{txt_all}}}"
    else:
        txt_mode = f"\\textbf{{{txt_mode}}}"

    # order: sys, Type, Holes, ΔHoles−Type, Instance, Non-Instance, ΔNonInst−Inst
    rows.append((sys, txt_all, txt_holes, txt_dh, txt_mode, txt_other, txt_do))

# 8. Emit LaTeX with deltas at columns 4 and 7
print(r"""\begin{table*}[ht]
\centering
\caption{Caption}
\label{tab:ode_deltas}
\begin{tabular}{ccccccc}
\toprule
\textbf{S} & \textbf{Type} & \textbf{Holes} & \textbf{$\Delta_{reduced-type}$} & \textbf{Instance} & \textbf{Non-Instance} & \textbf{$\Delta_{non-inst−inst}$} \\
\cmidrule(lr){2-2} \cmidrule(lr){3-3} \cmidrule(lr){4-4} \cmidrule(lr){5-5} \cmidrule(lr){6-6} \cmidrule(lr){7-7}
 & \textit{mean $\pm$ std} & \textit{mean $\pm$ std} & & \textit{mean $\pm$ std} & \textit{mean $\pm$ std} & \\
\midrule""")
for sys, a, h, dh, inst, oth, do in rows:
    print(f"{sys} & {a} & {h} & {dh} & {inst} & {oth} & {do} \\\\")
print(r"""\bottomrule
\end{tabular}
\end{table*}""")
